1. high cor over 0.8 with highest h2g
2. blood biochemistry(testoterone, IGF-1)
3. sex specific analysis between left_ear-right_ear vs our hip phenotypes

In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import pearsonr
import math
import pydicom
from show_keypoints import show_keypoints
import matplotlib.cm as cm
import scipy.stats as stats
from PIL import Image, ImageDraw
import os
import re
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

## All phenotypes correlation

In [ ]:
hip_pheno_flt = pd.read_csv('key_results/hip_pheno_23_norm_height_flt_eid_flt.csv', index_col=0); hip_pheno_flt

#### Left and right average

In [ ]:
df = hip_pheno_flt.copy()
df = df.iloc[:, 3:]
df

In [ ]:
averages_df = pd.DataFrame(index=df.index)

def calculate_average(df, col1, col2):
    if col2 is None:
        return
    avg_col_name = f'avg_{col1}_and_{col2}'
    averages_df[avg_col_name] = (df[col1] + df[col2]) / 2

def match_columns(column):
    pattern1 = re.compile(r'^(.*?)_(left|right)2(.*?)_(left|right)$')
    pattern2 = re.compile(r'^(.*?)_(left|right)2(.*?)$')
    pattern3 = re.compile(r'^(.*?)2(.*?)_(left|right)$')
    match1 = pattern1.match(column)
    match2 = pattern2.match(column)
    match3 = pattern3.match(column)
    return match1, match2, match3

handled_columns = []

for col in df.columns:
    match1, match2, match3 = match_columns(col)
    
    if match1:
        base_name, side1, target_name, side2 = match1.groups()
        # remove diagonal
        if side1 != side2 and base_name != target_name:
            if "divide" not in col:
                continue
        other_side1 = 'left' if side1 == 'right' else 'right'
        other_side2 = 'left' if side2 == 'right' else 'right'
        other_col = f'{base_name}_{other_side1}2{target_name}_{other_side2}'
        
        if other_col in df.columns:
            if col not in handled_columns and other_col not in handled_columns:
                calculate_average(df, col, other_col)
                handled_columns.append(col)
                handled_columns.append(other_col)
        else:
            averages_df[col] = df[col]
            
    elif match2:
        base_name, side1, target_name = match2.groups()
        other_side = 'left' if side1 == 'right' else 'right'
        other_col = f'{base_name}_{other_side}2{target_name}'
        
        if other_col in df.columns:
            if col not in handled_columns and other_col not in handled_columns:
                calculate_average(df, col, other_col)
                handled_columns.append(col)
                handled_columns.append(other_col)
        else:
            averages_df[col] = df[col]
            
    elif match3:
        base_name, target_name, side2 = match3.groups()
        other_side = 'left' if side2 == 'right' else 'right'
        other_col = f'{base_name}2{target_name}_{other_side}'
        
        if other_col in df.columns:
            if col not in handled_columns and other_col not in handled_columns:
                calculate_average(df, col, other_col)
                handled_columns.append(col)
                handled_columns.append(other_col)
        else:
            averages_df[col] = df[col]
    else:
        averages_df[col] = df[col]

In [ ]:
averages_df

In [ ]:
# left and right angle averages
averages_df['acetabular_inclination'] = (averages_df['acetabular_inclination_left'] + averages_df['acetabular_inclination_right']) / 2

# remove two angles columns
averages_df = averages_df.drop(columns=['acetabular_inclination_left', 'acetabular_inclination_right'])

In [ ]:
averages_df.to_csv('key_results/hip_pheno_lr_averages.csv'); averages_df

##### All phenotypes correlation plot

In [ ]:
# Calculate pairwise correlations
correlations = averages_df.corr()

# Calculate pairwise p-values
p_values = pd.DataFrame(index=df.columns, columns=df.columns)
for col1 in df.columns:
    for col2 in df.columns:
        p_values.loc[col1, col2] = pearsonr(df[col1], df[col2])[1]

# Convert p-values DataFrame elements to float
p_values = p_values.astype(float)

In [ ]:
correlations.to_csv('key_results/cor_all_phenos.csv')

In [ ]:
correlations

In [ ]:
# Create a custom colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g = sns.clustermap(correlations, cmap=cmap, method='average', metric='euclidean', figsize=(70, 70))

# Reorder the correlations and p_values based on the clustering
reordered_correlations = correlations.iloc[g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind]
reordered_p_values = p_values.iloc[g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind]

# Plot the heatmap with the reordered correlations
ax = g.ax_heatmap
ax.clear()
sns.heatmap(reordered_correlations, cmap=cmap, square=True, linewidths=0.5, cbar=False, ax=ax)

plt.savefig("out_fig/cor_all_phenos.pdf", bbox_inches='tight')

# Display the plot
plt.tight_layout()
plt.show()

##### Phenotype plot for manually selected phenotypes and five new phenotyps added on May 17, 2023

In [ ]:
hip_pheno_flt = pd.read_csv('key_results/hip_pheno_23_norm_height_flt_eid_flt.csv', index_col=0); hip_pheno_flt

In [ ]:
# Manually selected phenotypes
selected_phenos = ['iliac_spine_left2iliac_spine_right',
                   'sacrum_left2sacrum_right',
                   'sciatic_notch_left2sciatic_notch_right',
                   'inferior_iliac_spine_left2inferior_iliac_spine_right',
                   'iliopubic_eminence_left2acetabular_inferior_left',
                   'iliopubic_eminence_right2acetabular_inferior_right',
                   'sacrum2pubic_tubercle',
                   'acetabular_inclination_left',
                   'acetabular_inclination_right',
                   'pubic_arch_angle',
                   'hip_height']

added_phenos = ['sacrum_left2sacrum_right_divide_iliac_spine_left2iliac_spine_right',
                'sciatic_notch_left2sciatic_notch_right_divide_iliac_spine_left2iliac_spine_right',
                'inferior_iliac_spine_left2inferior_iliac_spine_right_divide_iliac_spine_left2iliac_spine_right',
                'angle_pubic_tubercle_iliac_spine_lr',
                'angle_pubic_arch_iliac_spine_lr',
                'sciatic_notch_left2inferior_iliac_spine_left',
                'sciatic_notch_right2inferior_iliac_spine_right',]

sub_pheno = hip_pheno_flt[selected_phenos + added_phenos]

# average of left and right
sub_pheno['iliopubic_eminence2acetabular_inferior'] = \
    (sub_pheno['iliopubic_eminence_left2acetabular_inferior_left'] + sub_pheno['iliopubic_eminence_right2acetabular_inferior_right']) / 2
sub_pheno = sub_pheno.drop(columns=['iliopubic_eminence_left2acetabular_inferior_left', 'iliopubic_eminence_right2acetabular_inferior_right'])

sub_pheno['acetabular_inclination'] = (sub_pheno['acetabular_inclination_left'] + sub_pheno['acetabular_inclination_right']) / 2
sub_pheno = sub_pheno.drop(columns=['acetabular_inclination_left', 'acetabular_inclination_right'])

sub_pheno['sciatic_notch2inferior_iliac_spine'] = \
    (sub_pheno['sciatic_notch_left2inferior_iliac_spine_left'] + sub_pheno['sciatic_notch_right2inferior_iliac_spine_right']) / 2
sub_pheno = sub_pheno.drop(columns=['sciatic_notch_left2inferior_iliac_spine_left', 'sciatic_notch_right2inferior_iliac_spine_right'])

In [ ]:
sub_pheno

In [ ]:
# Calculate pairwise correlations
correlations = sub_pheno.corr()

# Create a custom colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g = sns.clustermap(correlations, cmap=cmap, method='average', metric='euclidean', figsize=(15, 15), annot=True, fmt=".2f")

x0, _y0, _w, _h = g.cbar_pos

g.ax_cbar.set_position([0.65, 0.25, 0.03, 0.1])
g.ax_cbar.set_title('$Pearsonr$')

# save 
plt.savefig("out_fig/cor_manual_phenos.pdf", bbox_inches='tight')

# Display the plot
plt.tight_layout()
plt.show()

##### Phenotypes correlation of male and female

In [ ]:
all_dcm_info = pd.read_csv("all_prediction/all_dcm_info.csv")[['image_id', 'p_sex']]
all_pheno = sub_pheno.merge(all_dcm_info, left_index= True, right_on='image_id', how='inner')

In [ ]:
pheno_m = all_pheno[all_pheno['p_sex'] == 'M']
pheno_f = all_pheno[all_pheno['p_sex'] == 'F']

In [ ]:
# Create a custom colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Calculate pairwise correlations
correlations_m = pheno_m.corr()

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g1 = sns.clustermap(correlations_m, cmap=cmap, method='average', metric='euclidean', figsize=(15, 15), annot=True, fmt=".2f")

g1.ax_cbar.set_position([0.6, 0.25, 0.03, 0.1])  # Adjust position of colorbar
g1.ax_cbar.set_title('Pearsonr')
g1.fig.suptitle("Male")

plt.figure()  # create a new figure for the second clustermap

# Calculate pairwise correlations
correlations_f = pheno_f.corr()

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g2 = sns.clustermap(correlations_f, cmap=cmap, method='average', metric='euclidean', figsize=(15, 15), annot=True, fmt=".2f")

g2.ax_cbar.set_position([0.6, 0.25, 0.03, 0.1])  # Adjust position of colorbar
g2.ax_cbar.set_title('Pearsonr')
g2.fig.suptitle("Female")

plt.show()


In [ ]:
pheno_m.columns

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(x='sciatic_notch_left2sciatic_notch_right', y='sacrum2pubic_tubercle', data=pheno_m, ax=ax[0], alpha = 0.3, color='steelblue')
r = pearsonr(pheno_m['sciatic_notch_left2sciatic_notch_right'], pheno_m['sacrum2pubic_tubercle'])[0]
ax[0].set_title('Male\n$r = {:.2f}$'.format(r))
ax[0].set_xlabel('sciatic notch left -> sciatic notch right (brown)')
ax[0].set_ylabel('sacrum -> pubic tubercle (red)')

sns.scatterplot(x='sciatic_notch_left2sciatic_notch_right', y='sacrum2pubic_tubercle', data=pheno_f, ax=ax[1], alpha = 0.3, color='indianred')
r = pearsonr(pheno_f['sciatic_notch_left2sciatic_notch_right'], pheno_f['sacrum2pubic_tubercle'])[0]
ax[1].set_title('Female\n$r = {:.2f}$'.format(r))
ax[1].set_xlabel('sciatic notch left -> sciatic notch right (brown)')
ax[1].set_ylabel('sacrum -> pubic tubercle (red)')

sns.scatterplot(x='sciatic_notch_left2sciatic_notch_right', y='sacrum2pubic_tubercle', data=all_pheno, ax=ax[2], alpha = 0.3, hue=all_pheno['p_sex'], palette=['steelblue', 'indianred', 'green'])
r = pearsonr(all_pheno['sciatic_notch_left2sciatic_notch_right'], all_pheno['sacrum2pubic_tubercle'])[0]
ax[2].set_title('Both\n$r = {:.2f}$'.format(r))
ax[2].set_xlabel('sciatic notch left -> sciatic notch right (brown)')
ax[2].set_ylabel('sacrum -> pubic tubercle (red)')

plt.tight_layout()

In [ ]:
correlations_diff = correlations_f - correlations_m

# Calculate pairwise correlations
correlations = sub_pheno.corr()

# Create a custom colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g = sns.clustermap(correlations_diff, cmap=cmap, method='average', metric='euclidean', figsize=(15, 15), annot=True, fmt=".2f")

x0, _y0, _w, _h = g.cbar_pos

g.ax_cbar.set_position([0.65, 0.25, 0.03, 0.1])
g.ax_cbar.set_title('$Pearsonr$')

# save 
# plt.savefig("out_fig/cor_manual_phenos.pdf", bbox_inches='tight')

# Display the plot
plt.tight_layout()
plt.show()

#### Group phenotypes based on their correlation

In [ ]:
correlations = pd.read_csv('key_results/cor_all_phenos.csv', index_col=0); correlations

In [ ]:
correlations_melt = correlations.reset_index().melt(id_vars='index', var_name='index2', value_name='correlation'); correlations_melt

In [ ]:
correlations_melt.to_csv('key_results/cor_all_phenos_melt.csv')

##### hierarchical clustering

In [ ]:
distance_matrix = 1 - correlations.abs()
condensed_matrix = squareform(distance_matrix)
Z = linkage(condensed_matrix, method='average')

# Plot the dendrogram
plt.figure(figsize=(12, 5))

# Add a horizontal line at the threshold
threshold = 0.2
plt.axhline(y=threshold, color='red', linestyle='--')

dendrogram(
    Z,
    labels=distance_matrix.index,
    leaf_rotation=90,
    color_threshold=threshold
)
plt.xlabel("Phenotypes")
plt.ylabel("Distance")

plt.savefig("out_fig/dendrogram_all_phenos.pdf", bbox_inches='tight')

##### get groups

In [ ]:
# Set specific threshold value
threshold = 0.7

# Obtain clusters using the specific threshold
clusters = fcluster(Z, threshold, criterion='distance')

# Create a DataFrame with cluster assignments
cluster_df = pd.DataFrame({'phenotype': distance_matrix.index, 'cluster': clusters})

# Create the output format as a 2D list of groups
grouped = cluster_df.groupby('cluster')['phenotype'].apply(list)
groups = [group for _, group in grouped.items()]

In [ ]:
len(groups), groups

In [ ]:
r = []
for group in groups:
    for pheno1 in group:
        for pheno2 in group:
            if pheno1 != pheno2:
                if correlations.at[pheno1, pheno2] <= 0.8:
                    r.append(correlations.at[pheno1, pheno2])
sorted(r)

#### Get phenotypes with highest h2g

##### Average left and right

In [ ]:
# load h2g for each phenotyps
h2g_df = pd.read_csv('key_results/h2g/h2g_20230505.csv', index_col=0)
h2g_dict = h2g_df.to_dict(orient='index') 

In [ ]:
len(h2g_dict.keys())

In [ ]:
hip_pheno_flt = pd.read_csv('key_results/hip_pheno_23_norm_height_flt_eid_flt.csv'); hip_pheno_flt

In [ ]:
pheno_cols = hip_pheno_flt.columns[4:]
len(pheno_cols)

In [ ]:
# make sure h2g keys are the same as phenotypes
for i in pheno_cols:
    if i not in h2g_dict.keys():
        print(i)

for i in h2g_dict.keys():
    if i not in pheno_cols:
        print(i)

In [ ]:
# make sure the order of h2g keys are the same as phenotypes

h2g_dict = {key: h2g_dict[key] for key in pheno_cols}

In [ ]:
averages_dict = {}

def calculate_average(dict, key1, key2):
    if key2 is None:
        return
    avg_key_name = f'avg_{key1}_and_{key2}'
    averages_dict[avg_key_name] = {'var': (h2g_dict[key1]['var'] + h2g_dict[key2]['var']) / 2,
                                   'se': (h2g_dict[key1]['se'] + h2g_dict[key2]['se']) / 2}

def match_columns(key):
    pattern1 = re.compile(r'^(.*?)_(left|right)2(.*?)_(left|right)$')
    pattern2 = re.compile(r'^(.*?)_(left|right)2(.*?)$')
    pattern3 = re.compile(r'^(.*?)2(.*?)_(left|right)$')
    match1 = pattern1.match(key)
    match2 = pattern2.match(key)
    match3 = pattern3.match(key)
    return match1, match2, match3

handled_keys = []

for key in h2g_dict.keys():
    match1, match2, match3 = match_columns(key)
    
    if match1:
        base_name, side1, target_name, side2 = match1.groups()
        # remove diagonal
        if side1 != side2 and base_name != target_name:
            if "divide" not in key:
                continue
        other_side1 = 'left' if side1 == 'right' else 'right'
        other_side2 = 'left' if side2 == 'right' else 'right'
        other_key = f'{base_name}_{other_side1}2{target_name}_{other_side2}'
        
        if other_key in h2g_dict.keys():
            if key not in handled_keys and other_key not in handled_keys:
                calculate_average(h2g_dict, key, other_key)
                handled_keys.append(key)
                handled_keys.append(other_key)
        else:
            averages_dict[key] = h2g_dict[key]
            
    elif match2:
        base_name, side1, target_name = match2.groups()
        other_side = 'left' if side1 == 'right' else 'right'
        other_key = f'{base_name}_{other_side}2{target_name}'
        
        if other_key in h2g_dict.keys():
            if key not in handled_keys and other_key not in handled_keys:
                calculate_average(h2g_dict, key, other_key)
                handled_keys.append(key)
                handled_keys.append(other_key)
        else:
            averages_dict[key] = h2g_dict[key]
            
    elif match3:
        base_name, target_name, side2 = match3.groups()
        other_side = 'left' if side2 == 'right' else 'right'
        other_key = f'{base_name}2{target_name}_{other_side}'
        
        if other_key in h2g_dict.keys():
            if key not in handled_keys and other_key not in handled_keys:
                calculate_average(h2g_dict, key, other_key)
                handled_keys.append(key)
                handled_keys.append(other_key)
        else:
            averages_dict[key] = h2g_dict[key]
    else:
        averages_dict[key] = h2g_dict[key]

In [ ]:
# left and right angle averages
averages_dict['acetabular_inclination'] = {"var":(averages_dict['acetabular_inclination_left']['var'] + averages_dict['acetabular_inclination_right']['var']) / 2,
                                           "se":(averages_dict['acetabular_inclination_left']['se'] + averages_dict['acetabular_inclination_right']['se']) / 2}

# remove two angles columns
averages_dict.pop('acetabular_inclination_left')
averages_dict.pop('acetabular_inclination_right')

In [ ]:
len(averages_dict.keys())

In [ ]:
len(averages_df.columns)

In [ ]:
# to test if the h2g avg left and right result align with the phenotypes
[i for i in averages_dict.keys() if i not in averages_df.columns]

In [ ]:
averages_dict

##### Get highest phenotypes

In [ ]:
# Get the phenotypes with the highest h2g in each group
highest_h2g_phenotypes = {}

for group in groups:
    max_h2g = -np.inf
    max_phenotype = None
    for phenotype in group:
        if phenotype in averages_dict and averages_dict[phenotype]['var'] > max_h2g:
            max_h2g = averages_dict[phenotype]['var']
            max_phenotype = phenotype
    if max_phenotype is not None:
        highest_h2g_phenotypes[max_phenotype] = {}
        highest_h2g_phenotypes[max_phenotype]['var'] = max_h2g
        highest_h2g_phenotypes[max_phenotype]['se'] = averages_dict[max_phenotype]['se']

In [ ]:
len(highest_h2g_phenotypes.keys())

In [ ]:
list(highest_h2g_phenotypes.keys())

In [ ]:
selected_pheno = pd.DataFrame(highest_h2g_phenotypes).T; selected_pheno

In [ ]:
selected_pheno.shape

In [ ]:
selected_pheno.to_csv('key_results/highest_h2g_phenotypes_in_groups.csv')

### Phenotype correlation plot

In [ ]:
hip_pheno_male_flt = pd.read_csv('key_results/hip_select_pheno_male_flt.csv')
hip_pheno_female_flt = pd.read_csv('key_results/hip_select_pheno_female_flt.csv')

In [ ]:
correlations_f = hip_pheno_female_flt.iloc[:, 4:].corr()

In [ ]:
# Create a custom colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Calculate pairwise correlations
correlations_m = hip_pheno_male_flt.iloc[:, 4:].corr()

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g1 = sns.clustermap(correlations_m, cmap=cmap, method='average', metric='euclidean', figsize=(10, 10), annot=True, fmt=".2f", vmin=-1, vmax=1)

g1.ax_cbar.set_position([0.1, 0.9, 0.03, 0.1])  # Adjust position of colorbar
g1.ax_cbar.set_title('Pearsonr')
g1.fig.suptitle("Male")
plt.tight_layout()
plt.savefig("out_fig/pheno_cor_male_clustermap.pdf", bbox_inches="tight")

plt.figure()  # create a new figure for the second clustermap

# Calculate pairwise correlations
correlations_f = hip_pheno_female_flt.iloc[:, 4:].corr()

# Perform hierarchical clustering and reorder the rows and columns of the correlation matrix based on the clustering
g2 = sns.clustermap(correlations_f, cmap=cmap, method='average', metric='euclidean', figsize=(10, 10), annot=True, fmt=".2f", vmin=-1, vmax=1)

g2.ax_cbar.set_position([0.1, 0.9, 0.03, 0.1])   # Adjust position of colorbar
g2.ax_cbar.set_title('Pearsonr')
g2.fig.suptitle("Female")
plt.tight_layout()
plt.savefig("out_fig/pheno_cor_female_clustermap.pdf", bbox_inches="tight")

### plot relationship between age and pelvic width

In [ ]:
eids = pd.read_csv('key_results/eids_from_emily.csv')['eid'].tolist()

hip_pheno_gwas_male = pd.read_csv('key_results/hip_select_pheno_cm_male_residual.csv')
hip_pheno_gwas_male.sort_values(by = ['eid', 'file_name'], ascending=[True, True], inplace=True)
hip_pheno_gwas_male.drop_duplicates(subset=['eid'], keep='first', inplace=True)
hip_pheno_gwas_male = hip_pheno_gwas_male[hip_pheno_gwas_male['eid'].isin(eids)]

hip_pheno_gwas_female = pd.read_csv('key_results/hip_select_pheno_cm_female_residual.csv')
hip_pheno_gwas_female.sort_values(by = ['eid', 'file_name'], ascending=[True, True], inplace=True)
hip_pheno_gwas_female.drop_duplicates(subset=['eid'], keep='first', inplace=True)
hip_pheno_gwas_female = hip_pheno_gwas_female[hip_pheno_gwas_female['eid'].isin(eids)]

print(f"Male dataframe shape: {hip_pheno_gwas_male.shape}")
print(f"Female dataframe shape: {hip_pheno_gwas_female.shape}")

In [ ]:
# merge age column
fid_info = pd.read_csv('../UKB_xray_image_info/fids/fid_disease/fid_info.csv')[['eid', 'age']]
hip_pheno_gwas_male = fid_info.merge(hip_pheno_gwas_male, on='eid', how='inner')
hip_pheno_gwas_female = fid_info.merge(hip_pheno_gwas_female, on='eid', how='inner')
print(f"Male dataframe shape: {hip_pheno_gwas_male.shape}")
print(f"Female dataframe shape: {hip_pheno_gwas_female.shape}")

In [ ]:
hip_pheno_gwas = pd.concat([hip_pheno_gwas_male, hip_pheno_gwas_female], axis=0)

In [ ]:
hip_pheno_gwas

In [ ]:
# Bin the ages into 5-year increments
hip_pheno_gwas['sex'] = hip_pheno_gwas['sex'].map({1: 'Male', 0: 'Female'})

bins = range(int(hip_pheno_gwas['age'].min()), int(hip_pheno_gwas['age'].max()) + 5, 5)
labels = [f'{i}-{i+5}' for i in bins[:-1]]
hip_pheno_gwas['age_bins'] = pd.cut(hip_pheno_gwas['age'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
# Create a boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(x='age_bins', y='pelvic_inlet_width', hue = 'sex', data=hip_pheno_gwas, palette={'Male': 'steelblue', 'Female': 'indianred'})
plt.xticks(rotation=90) # This ensures that your x-axis labels don't overlap
plt.title('Pelvic Inlet Width by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Pelvic Inlet Width')
plt.tight_layout()
plt.show()

In [ ]:
# Create a boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(x='age_bins', y='pelvic_width', hue = 'sex', data=hip_pheno_gwas, palette={'Male': 'steelblue', 'Female': 'indianred'})
plt.xticks(rotation=90) # This ensures that your x-axis labels don't overlap
plt.title('Pelvic Width by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Pelvic Width')
plt.tight_layout()
plt.show()